# 03 · The math: feature benefit, interference and the sparsity decomposition

**Paper:** "Mathematical Understanding", pp. 12–14, and the start of "Superposition as a Phase
Change", pp. 14–15.
**Library:** `superposition.theory` (every number here is exact, or a Monte-Carlo check of an
exact number).

**Learning goals**

1. Split the linear model's loss into *feature benefit* and *interference* (p. 13), and see why
   the linear model never gains from superposition.
2. Split the ReLU output model's loss by the number of active features,
   L = Σₖ pᵏ(1 − p)ⁿ⁻ᵏ Lₖ, and read the zero-input term L₀ and the 1-sparse term L₁ (p. 13).
3. See why the ReLU makes negative interference free, and how a negative bias turns small
   positive interference into none, at a price (p. 13).
4. Compute the exact expected loss of a whole model and minimize it over a family of weights:
   the "toy model of the toy model" behind the phase diagrams (pp. 14–15, notebook 04).

Nothing is trained; the notebook runs in well under a minute.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import math

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from superposition import metrics, theory
from superposition.data import IndependentSparse
from superposition.models import TiedModel
from superposition.plotting.phase import family_loss_figure
from superposition.train import evaluate_loss
from superposition.utils import make_generator


def model_with(kind: str, W, b) -> TiedModel:
    """A one-instance model with the given weights (hidden × feat) and biases (feat,)."""
    W = torch.as_tensor(W, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    model = TiedModel.from_kind(kind, W.shape[1], W.shape[0], 1)
    with torch.no_grad():
        model.W.copy_(W[None])
        model.b.copy_(b[None])
    return model


def sampled_loss(model: TiedModel, p: float, importance, n_samples: int = 400_000) -> float:
    """Monte-Carlo estimate of the expected loss on fresh samples (what training minimizes)."""
    dist = IndependentSparse(p, model.n_inst, model.n_features)
    importance = torch.as_tensor(importance, dtype=torch.float32)
    return float(evaluate_loss(model, dist, importance, n_samples, make_generator(0))[0])

## 1. The linear model: benefit against interference (p. 13)

The linear model reconstructs x′ = WᵀWx + b. Output i reads its own feature with coefficient
‖Wᵢ‖² and every other feature j with Wᵢ·Wⱼ. With independent features of variance
vⱼ = p/3 − p²/4 and mean μⱼ = p/2 (our data: 0 with probability S, else U[0, 1]), the expected
squared error of output i is exactly

  vᵢ (1 − ‖Wᵢ‖²)²  +  Σ_{j≠i} vⱼ (Wᵢ·Wⱼ)²  +  (μᵢ − Σⱼ (WᵀW)ᵢⱼ μⱼ − bᵢ)².

The first term is the **feature benefit** still missing (zero when ‖Wᵢ‖ = 1), the second is
**interference**, and the third vanishes for the best bias. This is the paper's equation, which
it states for unit-variance Gaussian inputs (footnote 10), with each term weighted by a variance.

Two features, one dimension, importances 1 and 0.7. **Predict:** which is cheaper for the linear
model: dropping the second feature, or storing both as an antipodal pair W = [1, −1]? What about
a shorter pair, W = [1, −1]/√2?

In [ ]:
p = 0.1
importance = torch.tensor([1.0, 0.7])
v = p / 3 - p**2 / 4
mu = torch.full((2,), p / 2)
rows = []
for name, W in [
    ("drop feature 1: W = [1, 0]", [[1.0, 0.0]]),
    ("antipodal pair: W = [1, −1]", [[1.0, -1.0]]),
    ("shorter pair: W = [1, −1]/√2", [[2**-0.5, -(2**-0.5)]]),
]:
    W = torch.tensor(W)
    b = mu - W.T @ W @ mu  # the best bias predicts what WᵀW leaves out of the mean
    terms = theory.linear_loss_decomposition(W[None], b[None], importance, p)
    rows.append(
        {
            "model": name,
            "feature benefit": float(terms.feature_benefit[0]) / v,
            "interference": float(terms.interference[0]) / v,
            "bias term": float(terms.bias[0]) / v,
            "exact loss": float(terms.total[0]) / v,
            "Monte Carlo": sampled_loss(model_with("linear", W, b), p, importance) / v,
        }
    )
print(f"losses in units of the feature variance v = p/3 − p²/4 = {v:.5f}")
pd.DataFrame(rows).round(3)

Dropping feature 1 costs I₁·v = 0.7 v. The full antipodal pair stores both features perfectly
but each one now reads the other at full strength: (I₀ + I₁)·v = 1.7 v of interference. The
best pair, W = [a, −a], minimizes (1 − a²)² + a⁴ at a² = 1/2 and still costs (I₀ + I₁)·v/2 =
0.85 v, more than dropping the less important feature unless the two importances are equal.
That is the paper's point (footnote 12): **a linear model never gains from superposition.** It
behaves like PCA and keeps the top m features.

## 2. The loss, split by the number of active features (p. 13)

For the ReLU output model x′ = ReLU(WᵀWx + b) the paper groups the expectation by how many
features are active: L = Σₖ pᵏ(1 − p)ⁿ⁻ᵏ Lₖ, where Lₖ adds up the expected loss over every set of
exactly k active features. **Predict:** at p = 0.01 with n = 5 features, what share of inputs has
at most one active feature?

In [ ]:
n = 5
shares = pd.DataFrame(
    {
        f"p = {p}": [math.comb(n, k) * w for k, w in enumerate(theory.k_sparse_weights(n, p))]
        for p in (1.0, 0.3, 0.1, 0.01)
    },
    index=pd.Index(range(n + 1), name="active features k"),
)
shares.round(4)  # each column sums to 1: the probability of exactly k active features

As S → 1 the k = 0 and k = 1 terms carry almost all the probability. So in the sparse regime the
model is governed by two terms:

* **L₀ = Σᵢ Iᵢ ReLU(bᵢ)²**, the loss on the all-zero input: a penalty on positive biases.
* **L₁ = Σᵢ ∫ Iᵢ (xᵢ − ReLU(‖Wᵢ‖² xᵢ + bᵢ))² dxᵢ + Σ_{i≠j} ∫ Iⱼ ReLU(Wⱼ·Wᵢ xᵢ + bⱼ)² dxᵢ**: when only
  feature i is on, its own output should reproduce it (benefit), and every other output j
  should stay at zero (interference).

Here is the p. 2 pentagon, built by hand: five unit directions 72° apart, scaled to
‖Wᵢ‖ = 1.1, with every bias at −0.2 (the values E00 finds, docs/REPRODUCTION.md). **Predict:**
which entries of the interference matrix are nonzero?

In [ ]:
angles = 2 * math.pi * torch.arange(5) / 5
W = 1.1 * torch.stack([angles.cos(), angles.sin()])  # (2 hidden, 5 features)
b = torch.full((5,), -0.2)
importance = 0.9 ** torch.arange(5.0)
terms = theory.one_sparse_decomposition(W[None], b[None], importance)
pentagon = model_with("relu_output", W, b)
L0_mc = metrics.loss_by_num_active(pentagon, 0, importance, 10, make_generator(0))
L1_mc = metrics.loss_by_num_active(pentagon, 1, importance, 400_000, make_generator(1))
print(f"L₀: exact {float(terms.zero[0]):.5f}, sampled {float(L0_mc[0]):.5f}")
print(f"L₁: exact {float(terms.one[0]):.5f}, sampled {float(L1_mc[0]):.5f}")
print("benefit term of each feature:", terms.benefit[0].numpy().round(5))
features = [f"feature {i}" for i in range(5)]
pd.DataFrame(
    terms.interference[0].numpy(),
    index=pd.Index(features, name="output hurt"),
    columns=pd.Index(features, name="active feature"),
).round(5)

Only neighbours on the pentagon interfere. Next-nearest features sit 144° apart, their dot
product is negative, and the ReLU clips a negative pre-activation to exactly zero error.

## 3. Negative interference is free, and a negative bias makes it more so (p. 13)

On a 1-sparse input the interference term of output j is ∫₀¹ ReLU(Wⱼ·Wᵢ x + bⱼ)² dx. **Predict:**
how negative must bⱼ be to make an overlap Wⱼ·Wᵢ = 0.3 free?

In [ ]:
rows = []
for bias in (0.0, -0.1, -0.2, -0.3):
    rows.append(
        {
            "bias bⱼ": bias,
            "overlap +0.3": float(theory.relu_square_1d(0.3, bias)),
            "overlap −0.3": float(theory.relu_square_1d(-0.3, bias)),
            "own error, ‖Wᵢ‖² = 1": float(theory.own_error_1d(1.0, bias)),
            "own error, ‖Wᵢ‖² = 1 − b": float(theory.own_error_1d(1.0 - bias, bias)),
        }
    )
pd.DataFrame(rows).round(5)

Negative overlap costs nothing at any bias ≤ 0. Positive overlap 0.3 costs nothing once b ≤ −0.3,
because 0.3x + b stays ≤ 0 on [0, 1]. The bias is not free, though: it also shifts the feature's
own readout (fourth column). Stretching the feature to ‖Wᵢ‖² = 1 − b gives the value 1 back at
x = 1 and leaves an error of at most b²/3 (last column). That is the compromise the p. 2
pentagon makes: ‖Wᵢ‖² + bᵢ ≈ 1.21 − 0.2 ≈ 1. The paper makes the same observation on p. 22:
pentagon features sit off the unit circle because of their negative biases.

## 4. The exact loss of a whole model, and the toy model of the toy model (pp. 14–15)

When every output reads at most one other feature, the expectation over all sparsity patterns
can be done exactly: `theory.relu_output_expected_loss` integrates the piecewise-quadratic terms
in closed form. That covers every model with n = 2 features in m = 1 dimension, which has three
natural solutions (p. 14): drop the extra feature (W ⊥ [0, 1]), give it the dimension and drop
the other (W ⊥ [1, 0]), or store both as an antipodal pair (W ⊥ [1, 1]). Each family has free
scalars (a weight scale and one bias per feature); `theory.optimize_family` minimizes over them.

**Predict:** with the extra feature 1.5× as important as the other, which family wins at density
1 − S = 0.05, and which at 1?

In [ ]:
extra = 1.5
rows = []
for p in (0.05, 1.0):
    for family in theory.N2M1_FAMILIES:
        solution = theory.optimize_family(family, [[1.0, extra]], [p])
        W, b = solution.W[0], solution.b[0]
        rows.append(
            {
                "density 1 − S": p,
                "family": family.name,
                "extra feature": family.label,
                "optimal loss": solution.loss[0],
                "Monte Carlo": sampled_loss(model_with("relu_output", W, b), p, [1.0, extra]),
                "W": W.round(3).tolist(),
                "b": b.round(3).tolist(),
            }
        )
table = pd.DataFrame(rows)
table["winner"] = (
    table.groupby("density 1 − S")["optimal loss"].transform("min") == table["optimal loss"]
)
table

At density 1 every input has both features on, so an antipodal pair always interferes and the
model gives the more important extra feature the dimension. At density 0.05 the two features
are rarely on together, so the pair costs little and stores both. Somewhere in between the
winning family changes.

Sweeping the density shows how: each family's optimal loss is a smooth curve, and the winner
switches where two curves cross. The minimum of the curves has a kink there, a discontinuity in
its derivative: a **first-order phase change** (p. 15).

In [ ]:
density = np.geomspace(0.01, 1.0, 40)
importances = [0.5, 1.5, 4.0]
diagram = theory.theoretical_phase_diagram(theory.N2M1_FAMILIES, importances, density)
display(
    family_loss_figure(
        density,
        np.stack([diagram.losses[:, k, :] for k in range(len(importances))]),
        [f.name for f in diagram.families],
        [f.label for f in diagram.families],
        importances,
        suptitle="n = 2, m = 1: each family's optimal exact loss",
    )
)

Losses are divided by p/3 − p²/4 so that one axis fits all densities (losses at different
densities measure different tasks, footnote 17). Notebook 04 turns these crossings into the
phase diagrams of pp. 15–16 and compares them with trained models.

## Exercises

**1.** Show that the bias term of the linear decomposition vanishes at b = (I − WᵀW)μ, and check
it on a random W with n = 6, m = 3.

<details><summary>Solution</summary>

The mean of output i's error is μᵢ − Σⱼ(WᵀW)ᵢⱼμⱼ − bᵢ; setting it to zero gives b = (I − WᵀW)μ, and
the error's second moment is then its variance alone.

```python
p = 0.3
W = torch.randn(1, 3, 6)
mu = torch.full((6,), p / 2)
b = mu - W[0].T @ W[0] @ mu
terms = theory.linear_loss_decomposition(W, b[None], torch.ones(6), p)
print(float(terms.bias[0]))  # ≈ 0
```
</details>

**2.** In a regular pentagon with unit norms, how negative must every bias be before a single
active feature causes no interference at all? Compare what that bias costs the active
feature's own output, with ‖Wᵢ‖ = 1 and with ‖Wᵢ‖² = 1 − b, with the interference it removes.

<details><summary>Solution</summary>

Neighbours overlap by cos 72° ≈ 0.309, so every bias must be ≤ −0.309.

```python
c = math.cos(2 * math.pi / 5)
print("interference removed, per active feature:", 2 * float(theory.relu_square_1d(c, 0.0)))
print("own error, unit norm:", float(theory.own_error_1d(1.0, -c)))
print("own error, stretched:", float(theory.own_error_1d(1.0 + c, -c)))
```

Two neighbours at ≈ 0.032 each: about 0.064 of interference removed. With unit norms the own
error (≈ 0.076) costs more than that, so a bias alone does not pay. Stretched to ‖Wᵢ‖² = 1.309
it costs ≈ 0.019 and does pay. Trained pentagons (E00) settle in between, with b ≈ −0.2 and
‖Wᵢ‖ ≈ 1.1: they keep a little interference for large activations
(x > 0.2 / (1.21 · 0.309) ≈ 0.53) in exchange for a cheaper own term.
</details>

**3.** For n = 2 the decomposition has three terms, L = S²L₀ + pS·L₁ + p²L₂. Take any W (1 × 2)
and b, compute L exactly with `theory.relu_output_expected_loss`, L₀ and L₁ with
`theory.one_sparse_decomposition`, and check the L₂ this implies against
`metrics.loss_by_num_active(model, 2, …)`.

<details><summary>Solution</summary>

```python
W, b, p, imp = torch.tensor([[1.05, -0.95]]), torch.tensor([-0.08, 0.02]), 0.3, torch.tensor([1.0, 0.6])
exact = theory.relu_output_expected_loss(W, b, imp.numpy(), p)
terms = theory.one_sparse_decomposition(W[None], b[None], imp)
w0, w1, w2 = theory.k_sparse_weights(2, p)
L2 = (exact - w0 * float(terms.zero[0]) - w1 * float(terms.one[0])) / w2
model = model_with("relu_output", W, b)
print(L2, float(metrics.loss_by_num_active(model, 2, imp, 400_000, make_generator(0))[0]))
```
</details>

**4.** At relative importance 1.5, find the density where the antipodal pair stops beating the
dedicated-dimension family, to three significant figures. Where does it sit on the p. 15
diagram?

<details><summary>Solution</summary>

Bisect on the sign of the loss difference (on a log scale of density):

```python
pair, dedicated = theory.N2M1_FAMILIES[2], theory.N2M1_FAMILIES[1]


def gap(p):
    return (theory.optimize_family(pair, [[1.0, 1.5]], [p]).loss[0]
            - theory.optimize_family(dedicated, [[1.0, 1.5]], [p]).loss[0])


lo, hi = 0.01, 1.0  # the pair wins at lo, loses at hi
for _ in range(30):
    mid = math.sqrt(lo * hi)
    lo, hi = (mid, hi) if gap(mid) < 0 else (lo, mid)
print(f"crossover at 1 − S ≈ {math.sqrt(lo * hi):.3g}")
```

It lies on the red/blue boundary of the theoretical diagram, to the right of 1×.
</details>